In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import fastf1
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

import warnings
warnings.filterwarnings("ignore")

In [2]:
# Enable FastF1 caching
import os
os.makedirs("f1_cache", exist_ok=True)
fastf1.Cache.enable_cache("f1_cache")

### 2025 Data

In [3]:
# Load 2025 Italian GP race session, lap and sector times
session_2025 = fastf1.get_session(2026, "Italy", "R")
session_2025.load()
laps_2025 = session_2025.laps[["Driver", "LapTime", "Sector1Time", "Sector2Time", "Sector3Time"]].copy()
laps_2025.dropna(inplace=True)

core           INFO 	Loading data for Italian Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Driver 12 completed the race distance 00:00.013000 before the recorded end of the session.
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 22 drivers: ['12', '63', '3', '1', '81', '44', '10', '41', '43', '22', '5', '27

In [4]:
# Convert times to seconds
for col in ["LapTime", "Sector1Time", "Sector2Time", "Sector3Time"]:
    laps_2025[f"{col} (s)"] = laps_2025[col].dt.total_seconds()

In [5]:
# Group by driver to get average sector times per driver
sector_times_2025 = laps_2025.groupby("Driver")[["Sector1Time (s)", "Sector2Time (s)", "Sector3Time (s)"]].mean().reset_index()

### Quali Data 2026

In [6]:
# Qualifying data for 2026 Italian GP (Monza)
# Time = lap from the last session each driver reached (Q3 for P1-10, Q2 for P11-16, Q1 for P17-22)
qualifying_2026 = pd.DataFrame({
    "Driver": ["GAS", "RUS", "PIA", "LEC", "HAM", "VER", "ANT", "COL", "NOR", "LIN",
               "BOR", "BEA", "HUL", "LAW", "SAI", "OCO",
               "TSU", "ALB", "BOT", "PER", "ALO", "STR"],
    "QualifyingTime (s)": [81.786, 81.846, 81.966, 82.004, 82.011, 82.070, 82.093, 82.220, 82.256, 82.286,
                           82.517, 82.756, 82.779, 82.821, 83.453, 83.454,
                           83.755, 84.356, 84.364, 84.595, 85.150, 85.222]
})